# Data cleaning
Input: `Ad_table (extra).csv` and `Price_table.csv`. Output: `cars_clean.csv` (for EDA), `train.csv` and `test.csv` (for modelling).

## 1. Load the data
Adverts table (with the extra spec columns) and the new-car price table.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ad = pd.read_csv("../data/Ad_table (extra).csv", low_memory=False)
price = pd.read_csv("../data/Price_table.csv")
ad.columns = ad.columns.str.strip()  # some column names start with a space
print("starting rows:", len(ad))

starting rows: 268255


## 2. Remove rows we can't use
Duplicate relistings, missing or placeholder prices, and an impossible registration year.

In [2]:
before = len(ad)
ad = ad.drop_duplicates(subset=ad.columns.drop("Adv_ID"))   # same car relisted
ad = ad.dropna(subset=["Price"])                            # "Uknown" price
ad = ad[~ad["Price"].isin([999999, 9999999])]               # placeholder prices
ad = ad[ad["Reg_year"] != 1900]                             # impossible registration year
print("rows removed:", before - len(ad), "| rows left:", len(ad))

rows removed: 2018 | rows left: 266237


## 3. Turn text columns into numbers
Strip the units ("1.6L", "50.4 mpg", "140*") so these columns become numeric.

In [3]:
ad["Runned_Miles"] = pd.to_numeric(ad["Runned_Miles"].str.replace(" mile", ""), errors="coerce")
ad["Engin_size"] = pd.to_numeric(ad["Engin_size"].str.replace("L", ""), errors="coerce")
ad["Annual_Tax"] = pd.to_numeric(ad["Annual_Tax"].str.replace("*", "").str.strip(), errors="coerce")
ad["Average_mpg"] = pd.to_numeric(ad["Average_mpg"].str.replace(" mpg", ""), errors="coerce")
ad["Top_speed"] = pd.to_numeric(ad["Top_speed"].str.replace(" mph", ""), errors="coerce")
ad[["Runned_Miles", "Engin_size", "Annual_Tax", "Average_mpg", "Top_speed"]].describe()

,Runned_Miles,Engin_size,Annual_Tax,Average_mpg,Top_speed
count,2.652030e+05,264201.000000,219941.000000,225553.000000,223883.000000
mean,4.828089e+04,1.959890,155.862686,50.902846,122.052045
std,4.184491e+04,9.396849,120.853261,15.806988,18.543991
min,0.000000e+00,0.100000,0.000000,9.400000,11.000000
25%,1.430000e+04,1.400000,30.000000,41.500000,109.000000
50%,3.949600e+04,1.800000,140.000000,50.400000,118.000000
75%,7.500000e+04,2.000000,195.000000,60.100000,131.000000
max,6.363342e+06,3500.000000,555.000000,471.000000,226.000000


## 4. Fix impossible values
Price copied into mileage, impossible mileages, engine sizes typed in cc, and 0 mm dimensions.

In [4]:
ad = ad[ad["Price"] != ad["Runned_Miles"]]             # price copied into mileage (e.g. the Kia Cee'd)
ad = ad[~(ad["Runned_Miles"] > 500000)]                 # impossible mileage (keeps missing ones)

ad.loc[ad["Engin_size"] > 10, "Engin_size"] = ad["Engin_size"] / 1000   # typed in cc, not litres
ad.loc[ad["Fuel_type"] == "Electric", "Engin_size"] = 0                  # electric cars have no engine size
ad.loc[ad["Top_speed"] < 50, "Top_speed"] = np.nan
ad.loc[ad["Door_num"] == 0, "Door_num"] = np.nan
for col in ["Wheelbase", "Height", "Length"]:
    ad.loc[ad[col] == 0, col] = np.nan                                   # 0 mm = not filled in
ad.loc[ad["Bodytype"] == "Manual", "Bodytype"] = np.nan                  # a gearbox, not a body type
print("rows left:", len(ad))

rows left: 266169


## 5. Group the fuel types
Merge the 13 fuel labels into 6 groups.

In [5]:
fuel_groups = {
    "Petrol": "Petrol",
    "Diesel": "Diesel",
    "Electric": "Electric",
    "Hybrid  Petrol/Electric": "Hybrid",
    "Petrol Hybrid": "Hybrid",
    "Hybrid  Diesel/Electric": "Hybrid",
    "Diesel Hybrid": "Hybrid",
    "Hybrid  Petrol/Electric Plug-in": "Plug-in Hybrid",
    "Petrol Plug-in Hybrid": "Plug-in Hybrid",
    "Hybrid  Diesel/Electric Plug-in": "Plug-in Hybrid",
    "Diesel Plug-in Hybrid": "Plug-in Hybrid",
    "Bi Fuel": "Other",
    "Petrol Ethanol": "Other",
}
ad["Fuel_type"] = ad["Fuel_type"].map(fuel_groups)
ad["Fuel_type"].value_counts(dropna=False)

Fuel_type
Diesel            131737
Petrol            126380
Hybrid              5028
Electric            1301
Plug-in Hybrid      1290
NaN                  398
Other                 35
Name: count, dtype: int64

## 6. Build the target
Retention = used price ÷ new entry price of the same model in its registration year.

In [6]:
df = ad.merge(price[["Genmodel_ID", "Year", "Entry_price"]],
              left_on=["Genmodel_ID", "Reg_year"], right_on=["Genmodel_ID", "Year"])
df = df.drop(columns="Year")
print("ads matched to a new price:", round(len(df) / len(ad) * 100, 1), "%")

df["age"] = df["Adv_year"] - df["Reg_year"]
df = df[df["age"] >= 0]
df["retention"] = df["Price"] / df["Entry_price"]
df["retention"].describe()

ads matched to a new price: 83.9 %


count    223161.000000
mean          0.569653
std           0.331286
min           0.005597
25%           0.314404
50%           0.538331
75%           0.779398
max           7.061085
Name: retention, dtype: float64

## 7. Remove extreme ratios
Drop retention below 0.02 or above 3.

In [7]:
print("retention below 0.02:", (df["retention"] < 0.02).sum())
print("retention above 3:", (df["retention"] > 3).sum())
df = df[(df["retention"] >= 0.02) & (df["retention"] <= 3)]
print("rows left:", len(df))

retention below 0.02: 47
retention above 3: 50
rows left: 223064


## 8. Log-transform the target
Cars lose a similar percentage of value each year, so log retention falls in a straight line with age.

In [8]:
df["log_retention"] = np.log(df["retention"])
print("skew of retention:", round(df["retention"].skew(), 2), "| skew of log retention:", round(df["log_retention"].skew(), 2))
df.groupby("age")[["retention", "log_retention"]].median().head(12).round(3)

skew of retention: 0.71 | skew of log retention: -0.92


,retention,log_retention
age,,
0.0,1.041,0.040
1.0,0.911,-0.094
2.0,0.782,-0.246
3.0,0.664,-0.410
4.0,0.583,-0.540
5.0,0.520,-0.654
6.0,0.454,-0.791
7.0,0.399,-0.919
8.0,0.357,-1.031


## 9. Save a copy for EDA
Saved before imputing and scaling, so the plots show real values.

In [9]:
df.to_csv("../data/cars_clean.csv", index=False)
print("saved", df.shape)

saved (223064, 28)


## 10. Impute missing values
Fill specs from the median of the same car model; missing categories become "Unknown".

In [10]:
spec_cols = ["Engin_size", "Engine_power", "Annual_Tax", "Wheelbase", "Height", "Width",
             "Length", "Average_mpg", "Top_speed", "Seat_num", "Door_num"]
print("missing before:", df[spec_cols].isna().sum().sum())
for col in spec_cols:
    df[col] = df[col].fillna(df.groupby("Genmodel_ID")[col].transform("median"))
print("missing after filling from the same model:", df[spec_cols].isna().sum().sum())

cat_cols = ["Maker", "Color", "Bodytype", "Gearbox", "Fuel_type"]
df[cat_cols] = df[cat_cols].fillna("Unknown")

missing before: 129728
missing after filling from the same model: 3159


## 11. Train/test split by car model
No car model appears in both train and test. Split before scaling to avoid leakage.

In [11]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["Genmodel_ID"]))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()
print("train:", len(train), "ads,", train["Genmodel_ID"].nunique(), "models")
print("test: ", len(test), "ads,", test["Genmodel_ID"].nunique(), "models")
print("models in both:", len(set(train["Genmodel_ID"]) & set(test["Genmodel_ID"])))

train: 173042 ads, 465 models
test:  50022 ads, 117 models
models in both: 0


## 12. Check which features are skewed

In [12]:
num_cols = ["age", "Runned_Miles", "Entry_price"] + spec_cols
print(train[num_cols].skew().round(1).sort_values())

Door_num        -1.5
Length          -0.4
Wheelbase       -0.3
Seat_num        -0.1
Width            0.1
age              0.6
Height           0.8
Runned_Miles     0.8
Top_speed        1.1
Annual_Tax       1.3
Engin_size       2.2
Engine_power     2.6
Entry_price      5.6
Average_mpg     12.0
dtype: float64


## 13. Log-transform the skewed features
Only features with skew above 2.

In [13]:
skewed = ["Engin_size", "Engine_power", "Entry_price", "Average_mpg"]
train[skewed] = np.log1p(train[skewed])
test[skewed] = np.log1p(test[skewed])
print(train[skewed].skew().round(1))

Engin_size      0.5
Engine_power    0.6
Entry_price     0.9
Average_mpg     0.1
dtype: float64


## 14. Scale and encode
Median-impute remaining gaps, standardise numeric features, one-hot encode categories. Fitted on train only.

In [14]:
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(min_frequency=100, handle_unknown="infrequent_if_exist", sparse_output=False), cat_cols),
], verbose_feature_names_out=False)
preprocess.set_output(transform="pandas")

X_train = preprocess.fit_transform(train)
X_test = preprocess.transform(test)
print("features:", X_train.shape[1])
print("missing values left:", X_train.isna().sum().sum(), X_test.isna().sum().sum())
X_train.head()

features: 96
missing values left: 0 0


,age,Runned_Miles,Entry_price,Engin_size,Engine_power,Annual_Tax,Wheelbase,Height,Width,Length,...,Gearbox_Automatic,Gearbox_Manual,Gearbox_infrequent_sklearn,Fuel_type_Diesel,Fuel_type_Electric,Fuel_type_Hybrid,Fuel_type_Petrol,Fuel_type_Plug-in Hybrid,Fuel_type_Unknown,Fuel_type_infrequent_sklearn
0,2.876659,0.232943,3.837744,4.278162,2.823404,1.392775,2.619206,-0.204994,1.494618,2.458485,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2.409349,-0.167004,3.887069,4.278162,2.823404,1.392775,2.619206,-0.204994,1.494618,2.458485,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2.175693,0.107959,3.887069,4.278162,2.551160,1.392775,2.619206,-0.204994,1.494618,2.458485,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,2.175693,-0.916906,3.917229,4.278162,2.823404,1.392775,2.619206,-0.204994,1.494618,2.458485,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.942038,0.274238,3.917229,4.278162,2.823404,1.392775,2.619206,-0.204994,1.494618,2.458485,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## 15. Save the model-ready data
`Adv_ID` and `Genmodel_ID` are kept for lookups and grouped CV, not as features.

In [15]:
keep = ["Adv_ID", "Genmodel_ID", "log_retention"]
train_ready = pd.concat([train[keep].reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
test_ready = pd.concat([test[keep].reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
train_ready.to_csv("../data/train.csv", index=False)
test_ready.to_csv("../data/test.csv", index=False)
print("saved train", train_ready.shape, "and test", test_ready.shape)

saved train (173042, 99) and test (50022, 99)
